# NTU 选课助手 — 毕业设计 Demo

可溯源的 NTU 课程信息问答与选课建议多 Agent 系统。

- 作者：[@YounglWei](https://github.com/YounglWei)
- 基于 [hello-agents](https://github.com/datawhalechina/hello-agents) 1.0.1
- 完整评测与失败归因见 `04-评估报告.md`


In [1]:
# 首次运行：安装依赖（取消下一行注释）
# !pip install -r requirements.txt

import sys
from pathlib import Path

# 让 notebook 能从项目根导入 src
ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))
else:
    raise RuntimeError("请在项目根目录（包含 src/ 的目录）启动 Jupyter")

from src.config import RAG_DB_PATH, ZHIPU_API_KEY

print("ZHIPU_API_KEY:", "已配置" if ZHIPU_API_KEY else "未配置，请复制 .env.example 为 .env 并填入 key")
print("RAG 索引:", "已存在" if RAG_DB_PATH.exists() else f"缺失（{RAG_DB_PATH}），请先运行 python scripts/build_index.py")


ZHIPU_API_KEY: 已配置
RAG 索引: 已存在


## 0. 无需 API key 的确定性演示

下面两部分不调用大模型，可直接运行：身份查表与 CCA 学位校验。


In [2]:
from src.identity import identify
from src.validators import DegreeAuditTool

print("身份识别：", identify("我是 CCA 新生"))
print()
print(DegreeAuditTool().check_au_limit(["MA6210", "EE6203"], option=1).text)


提示：推荐安装 Python 3.13；当前 Python 3.12 可以继续运行。


身份识别： IdentityResult(status='confirmed', level='graduate', programme='MSc Computer Control & Automation', matched_signal='项目词→MSc Computer Control & Automation', claim=None, candidates=[])

按 Option 1（10 门课 30 AU，无论文） 校验：
  通选：0 门 0 AU（上限 18 AU）——符合
  专选：1 门 3 AU（下限 12 AU）——不足
  专选门数：1 门（要求 ≥4 门）——不足
总体结论：存在不满足项，见上。
以下课程不在 CCA 研究生课库内（本科课或其他项目课），是否可计入学位请以项目页/学院规定为准：MA6210
注：本工具只校验专选门数与 AU 分布，不校验总门数（Option 1 需 10 门；Option 2 需 8 门 + 论文 6 AU）与论文要求；完整学位合规请以项目页为准。
规则来源：https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-computer-control-automation


## 1. 三 Agent 组装

- Planner：`src/team.py` `generate_plan`（提示词在 `src/prompts.py`）
- Retrieval：`ReActAgent` + `rag_search` + 4 个校验工具
- Reviewer：`src/team.py` 编排 + `src/guards.py` 机械审查

下面构造编排对象，并列出 Retrieval 实际注册的工具。


In [3]:
import contextlib
import io
from concurrent.futures import ThreadPoolExecutor

from src.team import CourseAdvisorTeam

if not ZHIPU_API_KEY or not RAG_DB_PATH.exists():
    raise RuntimeError("请先配置 .env 并运行 python scripts/build_index.py")

questions = [
    "SC2001 是几 AU？哪些学期开课？",
    "我是 CCA 研究生，课程列表有哪些？",
    "CS1010S 这门课讲什么？",
]


def run_demo():
    # Jupyter 内核已有运行中的事件循环，框架同步入口会拒绝；
    # 放到工作线程里构造并运行，避免 asyncio 冲突与 SQLite 跨线程问题。
    with contextlib.redirect_stdout(io.StringIO()):
        team = CourseAdvisorTeam(verbose=False)
        tools = team.retrieval.list_tools()
        answers = [(q, team.run(q)) for q in questions]
    return tools, answers


with ThreadPoolExecutor(max_workers=1) as pool:
    tools, answers = pool.submit(run_demo).result()
print("Retrieval 工具：", tools)


Retrieval 工具： ['rag_read', 'rag_read_document', 'rag_search', 'check_au_limit', 'check_prerequisites', 'check_schedule_conflict', 'check_specialized_electives']


## 2. 真实问答演示

运行 3 条问题：属性查询、CCA 列表、库外课码。全部走完整链路（Planner → Retrieval → Polish → Reviewer）。


In [4]:
for question, answer in answers:
    print("=" * 72)
    print("用户:", question)
    print(answer)


用户: SC2001 是几 AU？哪些学期开课？
（课程信息全部来自 NTU 官方课程库，未使用模型自身知识，来源见文末）

SC2001 是一门 3.0 AU 的课程，属于算法设计与分析领域。该课程在 2026 学年的第一学期和第二学期开课。

来源：
- [SC2001 课程详情](https://wis.ntu.edu.sg/webexe/owa/aus_subj_cont.main)
用户: 我是 CCA 研究生，课程列表有哪些？
已理解身份：研究生 · CCA（本次按此检索）

（课程信息全部来自 NTU 官方课程库，未使用模型自身知识，来源见文末）

以下是 CCA 研究生课程列表：

1. 课程代码：EE6009，课程名称：EE6009 GRADUATE PROFESSIONAL INTERNSHIP，学分：3.0，类别：通选，学期：未列出（请参考专业页面）。
   [课程详情](https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-computer-control-automation)

2. 课程代码：EE6228，课程名称：EE6228 PROCESS MODELING & SCHEDULING，学分：3.0，类别：通选，学期：未列出（请参考专业页面）。
   [课程详情](https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-computer-control-automation)

3. 课程代码：EE6406，课程名称：EE6406 ANALYTIC & ENSEMBLE MACHINE LEARNING，学分：3.0，类别：通选，学期：未列出（请参考专业页面）。
   [课程详情](https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-computer-control-automation)

4. 课程代码：EE6111，课程名称：EE6111 5G COMMUNICATION & BEYOND，学分：未列出，类别：通选，学期：未列出（请参考专业页面）。
   [

## 预期结果

- `SC2001`：3.0 AU，2026 Sem 1/2，附 WIS 来源
- `CCA` 课程列表：3-5 门研究生课，附 CCA 项目页来源
- `CS1010S`：明说未查到，不拿相近课程硬答

## 评测结果（真实跑分）

| 版本 | 判据 | 准确率 | 单轮中位 |
|---|---|---|---|
| D4 基线 | v1 预注册 | 12/20 = 60.0% | 28.01s |
| D4.1 修复 | v1 预注册 | 17/20 = 85.0% | 25.44s |
| D4.2 判据修订 | v2 | 19/20 = 95.0% | 33.41s |

> v1 与 v2 不可直接比较；逐条归因见 `04-评估报告.md`。

## 已知限制

- 本科来源 URL 为 WIS 目录入口页，不是逐课永久链接；回答附课码，便于回目录检索。
- Robotics 13 门无课程简介；本科仅 CCDS 学院；研究生课表官方不公开。
- `glm-4-flash` 免费档偶发长响应与单次字段遗漏；LLM 时限已调到 240s。
- 不做自动选课/抢课。


In [5]:
# 可选：检索金标与端到端评估（需 API key，耗时较长）
# !python scripts/eval_retrieval.py
# !python scripts/eval_e2e.py --only Q01,Q02,Q15,Q17


## 许可证

本项目采用 [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) 许可。
课程数据版权归 Nanyang Technological University (NTU)，每张卡片保留官方 `source_url`；本项目仅用于学习/毕业设计。
